# 01 - DFlash draft-then-verify

**학습 목표**: block draft가 여러 token을 제안하고 target이 longest valid prefix를 받아들이는 speculative decoding을 구현합니다.

**실행 방법**: Python 3/Jupyter에서 cell을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 기능만 사용합니다.

실제 diffusion draft나 sampling distribution 증명이 아닌 deterministic toy입니다.

In [ ]:
# 문자열 slice를 candidate block으로 써 draft-verify 흐름을 가장 작은 형태로 관찰합니다.
def draft_block(target, pos, block_size):
    draft = list(target[pos:pos + block_size])
    # 두 번째 draft block마다 일부 후보를 틀리게 만들어 rejection을 관찰합니다.
    if (pos // block_size) % 2 and len(draft) > 3:
        draft[3] = '?'
    return draft

def generate_with_verification(target, block_size=8):
    output = []
    target_forwards = 0
    acceptance = []
    while len(output) < len(target):
        pos = len(output)
        draft = draft_block(target, pos, block_size)
        target_forwards += 1
        accepted = 0
        for proposal, truth in zip(draft, target[pos:]):
            if proposal != truth:
                break
            output.append(proposal)
            accepted += 1
        if len(output) < len(target) and accepted < len(draft):
            output.append(target[len(output)])  # target의 올바른 bonus token
        acceptance.append(accepted)
    return ''.join(output), target_forwards, acceptance

truth = '<table><tr><td>OCR</td></tr></table>'
prediction, forwards, accepted = generate_with_verification(truth, block_size=8)
print('prediction:', prediction)
print('target forwards:', forwards, 'AR forwards:', len(truth))
print('accepted prefix lengths:', accepted)
assert prediction == truth
assert forwards < len(truth)


DFlash는 실제로 block size 16, 평균 effective acceptance 약 8.36-8.89를 보고합니다. candidate가 거절돼도 target 검증으로 correctness를 유지하는 것이 direct diffusion decoder와의 핵심 차이입니다.